In [1]:
import numpy as np
import math
import pandas as pd
from scipy.stats import mode

def shapley_kernel_weight(num_features, subset_size, normalize=False):
    """
    Calculate the Shapley kernel weight for a subset of a given size.

    Args:
        num_features (int): Total number of features.
        subset_size (int): Number of features in the subset.
        normalize (bool): Whether to normalize the weight to sum to 1.

    Returns:
        float: Weight for the subset size.
    """
    # Ensure subset size is valid
    if subset_size == 0 or subset_size == num_features:
        return 0
    weight = (num_features - 1) / (math.comb(num_features, subset_size) * subset_size * (num_features - subset_size))
    
    # Normalize weight if specified
    if normalize:
        total_combinations = 2 ** num_features - 2  # Exclude empty set and full set
        weight /= total_combinations

    return weight


def sample_and_weight(X, num_samples="auto", normalize=False, weighted_sampling=False):
    """
    Sample subsets and compute Shapley kernel weights.

    Args:
        X (np.ndarray): Input data, shape (n_samples, n_features).
        num_samples (int or str): Number of subsets to sample. Use "auto" to calculate based on the formula.
        normalize (bool): Whether to normalize the weights.
        weighted_sampling (bool): Whether to sample subsets based on weights.

    Returns:
        np.ndarray: Matrix of sampled subsets (binary masks).
        np.ndarray: Array of corresponding weights.
    """
    num_features = X.shape[1]

    # Set default number of samples
    if num_samples == "auto":
        num_samples = 2 * num_features + 2048

    subsets = []
    weights = []

    # Precompute weights for all subset sizes if using weighted sampling
    if weighted_sampling:
        subset_weights = [shapley_kernel_weight(num_features, size, normalize) for size in range(1, num_features)]
        cumulative_weights = np.cumsum(subset_weights)
        cumulative_weights /= cumulative_weights[-1]  # Normalize to sum to 1

    for _ in range(num_samples):
        if weighted_sampling:
            # Sample subset size based on weights
            rand = np.random.random()
            subset_size = np.searchsorted(cumulative_weights, rand) + 1
        else:
            # Uniformly sample subset size
            subset_size = np.random.randint(1, num_features)
        
        # Generate a binary mask for the subset
        subset = np.zeros(num_features, dtype=int)
        subset[:subset_size] = 1
        np.random.shuffle(subset)

        # Compute the Shapley kernel weight
        weight = shapley_kernel_weight(num_features, subset_size, normalize)

        subsets.append(subset)
        weights.append(weight)

    return np.array(subsets), np.array(weights)


def prepare_data_subsets(sample_row, background_data, subsets, use_random=False):
    """
    Generate subsets with data substitution based on subsets and background data using matrix operations.
    A feature is treated as categorical if it has only 2 unique values (e.g., {0, 1}).

    Args:
        sample_row (pd.Series or np.ndarray): The single row of real data to explain.
        background_data (pd.DataFrame or np.ndarray): Background data used for substitution.
        subsets (np.ndarray): Binary subset matrix (1 = real data, 0 = background data).
        use_random (bool): 
            - If False, use mean (continuous) and mode (categorical) for substitution.
            - If True, randomly sample background data independently for each subset row.

    Returns:
        np.ndarray: Substituted data matrix where each row is a generated subset.
    """
    # Convert inputs to NumPy arrays if they are pandas DataFrames or Series
    if isinstance(sample_row, pd.Series):
        sample_row = sample_row.values
    if isinstance(background_data, pd.DataFrame):
        background_data = background_data.values

    num_features = background_data.shape[1]
    num_subsets = subsets.shape[0]

    # Step 1: Create a real value matrix by repeating the sample row
    real_value_matrix = np.tile(sample_row, (num_subsets, 1))

    # Step 2: Create background substitution matrix
    if use_random:
        # Random sampling: Generate a random sample for each row in subsets
        random_indices = np.random.randint(0, background_data.shape[0], size=num_subsets)
        background_substitution = background_data[random_indices, :]
    else:
        # Mean for continuous features and mode for categorical features
        background_substitution = []
        for i in range(num_features):
            unique_vals = np.unique(background_data[:, i])
            if len(unique_vals) == 2:  # If feature is categorical
                substitution_value = mode(background_data[:, i], keepdims=True).mode[0]
            else:  # Continuous
                substitution_value = np.mean(background_data[:, i])
            background_substitution.append(substitution_value)
        background_substitution = np.array(background_substitution)
        # Correctly repeat the substitution row for all subsets
        background_substitution = np.tile(background_substitution, (num_subsets, 1))

    # Step 3: Combine using matrix multiplication
    substituted_data = real_value_matrix * subsets + background_substitution * (1 - subsets)

    return substituted_data

Weighted Sampling with Auto Samples:
Subsets (Matrix):
[[1 1 1 0 1]
 [0 1 1 1 1]
 [1 1 0 1 0]
 ...
 [1 1 1 1 0]
 [0 1 1 0 1]
 [0 1 0 0 0]]

Weights (Vector):
[0.2        0.2        0.06666667 ... 0.2        0.06666667 0.2       ]

Uniform Sampling with Auto Samples:
Subsets (Matrix):
[[1 1 0 1 1]
 [0 1 0 0 0]
 [1 1 1 0 1]
 ...
 [0 0 1 1 1]
 [0 1 1 0 1]
 [1 1 1 0 1]]

Weights (Vector):
[0.2        0.2        0.2        ... 0.06666667 0.06666667 0.2       ]


In [3]:



# Example Usage
# Sample row to explain
sample_row = pd.Series([10.0, 1, 50.0, 0])  # Example real data (continuous + categorical)

# Background data
background_data = pd.DataFrame({
    'Feature1': [12.0, 15.0, 14.0, 13.0],  # Continuous
    'Feature2': [1, 0, 1, 0],              # Categorical (only 2 unique values)
    'Feature3': [60.0, 55.0, 50.0, 65.0],  # Continuous
    'Feature4': [0, 1, 0, 1]               # Categorical (only 2 unique values)
})

# Subsets (binary mask matrix)
subsets = np.array([
    [1, 0, 1, 0],  # Use Feature1 and Feature3 from sample_row, replace others with background
    [0, 1, 0, 1],  # Use Feature2 and Feature4 from sample_row, replace others with background
    [1, 1, 0, 0],  # Use Feature1 and Feature2 from sample_row, replace others with background
])

# Get substituted data using mean/mode for substitution
substituted_data_mean_mode = prepare_data_subsets(sample_row, background_data, subsets, use_random=False)
print("Substituted Data (Mean/Mode):")
print(substituted_data_mean_mode)

# Get substituted data using random sampling for substitution
substituted_data_random = prepare_data_subsets(sample_row, background_data, subsets, use_random=True)
print("\nSubstituted Data (Random Sampling):")
print(substituted_data_random)


Substituted Data (Mean/Mode):
[[10.   0.  50.   0. ]
 [13.5  1.  57.5  0. ]
 [10.   1.  57.5  0. ]]

Substituted Data (Random Sampling):
[[10.  1. 50.  0.]
 [13.  1. 65.  0.]
 [10.  1. 60.  0.]]
